In [0]:
%pip uninstall -y datasets huggingface_hub
%pip install datasets==2.18.0 huggingface_hub==0.23.0
dbutils.library.restartPython()   # restart Python to pick up the new versions

In [0]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

football = dataset.filter(lambda x: "football" in x["text"].lower() or "soccer" in x["text"].lower())

df_pandas = pd.DataFrame(football.take(100000))
df_spark = spark.createDataFrame(df_pandas)

# Save to Delta for later
df_spark.write.mode("overwrite").format("delta").save("/mnt/delta/football_articles")

In [0]:
%pip install wikipedia
%pip install lxml
dbutils.library.restartPython()   # restart the Python kernel to load the new library

In [0]:
import wikipedia

# ---- Mandatory: set a descriptive User-Agent ----
wikipedia.set_user_agent("MyFootballBot/1.0 (myemail@example.com)")

# Now fetch the page
page = wikipedia.page("Cristiano Ronaldo")
full_text = page.content

# Show a snippet
print(full_text)
print(f"... total length: {len(full_text)} characters")

In [0]:
import requests
import pandas as pd

headers = {"User-Agent": "FootballBot/1.0 (youremail@example.com)"}

params = {
    "action": "parse",
    "page": "Cristiano Ronaldo",
    "prop": "text",
    "format": "json"
}
resp = requests.get("https://en.wikipedia.org/w/api.php", params=params, headers=headers)
html = resp.json()["parse"]["text"]["*"]

tables = pd.read_html(html)


In [0]:
tables

In [0]:
import requests
import os

# 1. Configure
page_title = "Cristiano Ronaldo"
volume_path = "/Volumes/workspace/default/football_data"
save_path = f"{volume_path}/cristiano_ronaldo.pdf"
headers = {
    "User-Agent": "DatabricksFootballBot/1.0 (your-email@example.com)"
}

# 2. Ensure the volume exists
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.football_data")

# 3. Build the REST API URL
url = f"https://en.wikipedia.org/api/rest_v1/page/pdf/{requests.utils.quote(page_title)}"

# 4. Request the PDF (stream=True for large files, but PDFs are small)
response = requests.get(url, headers=headers, stream=True)

# 5. Check success and save
if response.status_code == 200:
    with open(save_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
    print(f"PDF saved to {save_path}")
else:
    print(f"Error {response.status_code}: {response.text}")

In [0]:
print(response)